# Animate Migration Tracks + ERA5 Weather

Test notebook for `animate_tracks_weather.py`.  
**Kernel**: `migration` conda env  
**Data**: `move_icarus_bats_solarnoon_daily.csv` — filtered to April 2026

### CSV loading notes
The file is exported from R with `write.csv()`.  Non-location rows (VeDBA only) have
`c(NA, NA)` in the geometry column — an unquoted comma that adds an extra field and shifts
all subsequent columns on those rows.  We work around this with `engine='python',
on_bad_lines='skip'` and use the pre-extracted `lon`/`lat` columns.  
Event timestamps live in `start_timestamp` for location rows (`timestamp` is NA for those).

In [5]:
import sys, subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from python.animate_tracks_weather import animate_tracks_weather

CSV      = Path(r"C:/Users/Edward/Dropbox/MPI/Noctule/Data/rdata/move_icarus_bats_solarnoon_daily.csv")
# UNC path must use backslashes on Windows for Python to resolve it correctly
ERA5_DIR = Path(r"\\10.0.16.7\grpdechmann\Postdoc-EdwardHurme\EnvData")
OUT_FILE = "migration_april2026.gif"

print("CSV exists:     ", CSV.exists())
print("ERA5 dir exists:", ERA5_DIR.exists())
print("single_levels/: ", (ERA5_DIR / "single_levels").exists())

CSV exists:      False
ERA5 dir exists: False
single_levels/:  False


In [2]:
# Install cfgrib if not already present (needed to read .grib files)
# Run once, then restart kernel
subprocess.run([sys.executable, "-m", "pip", "install", "cfgrib"], check=True)

CompletedProcess(args=['C:\\Users\\Edward\\miniconda3\\envs\\gee\\python.exe', '-m', 'pip', 'install', 'cfgrib'], returncode=0)

In [3]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
# engine='python' + on_bad_lines='skip' silently drops the ~362k VeDBA-only rows
# whose geometry column has unquoted c(NA, NA) (adds an extra comma, breaking the row).
# The remaining ~57k rows are location fixes with real lon/lat values.
df_raw = pd.read_csv(CSV, index_col=0, engine='python', on_bad_lines='skip')

# Keep only rows with actual coordinates
df_loc = df_raw.dropna(subset=['lon', 'lat']).copy()

# Event time is in start_timestamp for location rows (timestamp col is NA here)
df_loc['ts'] = pd.to_datetime(df_loc['start_timestamp'], errors='coerce', utc=True)

print(f"Total location rows: {len(df_loc)}")
print(f"Date range:          {df_loc['ts'].min().date()} – {df_loc['ts'].max().date()}")
print(f"Years:               {sorted(df_loc['ts'].dt.year.dropna().unique().astype(int))}")

Total location rows: 57405
Date range:          2022-04-14 – 2026-05-06
Years:               [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


## Data preparation

Helper that filters `df_loc` to a season (default: spring = Mar–May) and renames columns
to the standard names expected by `animate_tracks_weather`.

In [4]:
def prep_season(df_loc, year, months=(3, 4, 5)):
    """Return a clean DataFrame filtered to *year* and *months* with standard column names."""
    df = df_loc[
        (df_loc['ts'].dt.year == year) &
        (df_loc['ts'].dt.month.isin(months))
    ].copy()
    df = df.drop(columns=['timestamp', 'individual_local_identifier'], errors='ignore')
    df = df.rename(columns={
        'lon':                    'location_long',
        'lat':                    'location_lat',
        'ts':                     'timestamp',
        'tag_local_identifier':   'individual_local_identifier',
    })
    n   = len(df)
    ind = df['individual_local_identifier'].nunique()
    sp  = df['species'].nunique() if 'species' in df.columns else '?'
    t0  = df['timestamp'].min().date()
    t1  = df['timestamp'].max().date()
    print(f"{year} spring (months {list(months)}): {n:,} fixes | {ind} individuals | "
          f"{sp} species | {t0} – {t1}")
    return df


# ── Build datasets ────────────────────────────────────────────────────────────
df_apr26  = prep_season(df_loc, 2026, months=(4,))        # April 2026 only
df_spr26  = prep_season(df_loc, 2026, months=(3, 4, 5))   # Spring 2026
df_spr25  = prep_season(df_loc, 2025, months=(3, 4, 5))   # Spring 2025

2026 spring (months [4]): 4,761 fixes | 93 individuals | 3 species | 2026-04-01 – 2026-04-30
2026 spring (months [3, 4, 5]): 6,259 fixes | 94 individuals | 3 species | 2026-03-19 – 2026-05-06
2025 spring (months [3, 4, 5]): 19,906 fixes | 201 individuals | 3 species | 2025-03-18 – 2025-05-31


In [5]:
# ── Static preview ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
species_colors = {'Nyctalus noctula': '#e41a1c', 'Nyctalus leisleri': '#377eb8',
                  'Nyctalus lasiopterus': '#4daf4a'}

for ind, grp in df_apr26.groupby('individual_local_identifier'):
    grp_s = grp.sort_values('timestamp')
    sp = grp_s['species'].iloc[0] if 'species' in grp_s.columns else 'unknown'
    c = species_colors.get(sp, '#999999')
    ax.plot(grp_s['location_long'], grp_s['location_lat'],
            '-', color=c, linewidth=0.6, alpha=0.6)
    ax.scatter(grp_s['location_long'].iloc[-1], grp_s['location_lat'].iloc[-1],
               color=c, s=20, zorder=3)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=v, label=k) for k, v in species_colors.items()], fontsize=8)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('April 2026 — all tracks (dot = last fix)')
plt.tight_layout()
plt.show()

<positron-console-cell-5>:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Check ERA5 coverage for April 2026

In [6]:
single_dir = ERA5_DIR / "single_levels"
print(f"single_levels/ exists: {single_dir.exists()}")

if single_dir.exists():
    files = sorted(single_dir.glob("era5_single_*"))
    print(f"\nAvailable files ({len(files)}):")
    for f in files:
        print(" ", f.name)

    # Accept either .nc or .grib
    needed_stems = ["era5_single_2026_04"]
    for stem in needed_stems:
        found = any((single_dir / f"{stem}{ext}").exists() for ext in (".nc", ".grib"))
        status = "✓" if found else "✗ MISSING — run download_era5.py for 2026/April"
        print(f"\n{status}  {stem}.*")
else:
    print("⚠  ERA5 directory not reachable")

single_levels/ exists: True

Available files (56):
  era5_single_2022_01.grib
  era5_single_2022_02.grib
  era5_single_2022_03.grib
  era5_single_2022_04.grib
  era5_single_2022_05.grib
  era5_single_2022_06.grib
  era5_single_2022_07.grib
  era5_single_2022_08.grib
  era5_single_2022_09.grib
  era5_single_2022_10.grib
  era5_single_2022_11.grib
  era5_single_2022_12.grib
  era5_single_2023_01.grib
  era5_single_2023_02.grib
  era5_single_2023_03.grib
  era5_single_2023_04.grib
  era5_single_2023_05.grib
  era5_single_2023_06.grib
  era5_single_2023_07.grib
  era5_single_2023_08.grib
  era5_single_2023_09.grib
  era5_single_2023_10.grib
  era5_single_2023_11.grib
  era5_single_2023_12.grib
  era5_single_2024_01.grib
  era5_single_2024_02.grib
  era5_single_2024_03.grib
  era5_single_2024_04.grib
  era5_single_2024_05.grib
  era5_single_2024_06.grib
  era5_single_2024_07.grib
  era5_single_2024_08.grib
  era5_single_2024_09.grib
  era5_single_2024_10.grib
  era5_single_2024_11.grib
  er

## April 2026 — comparison renders

**Version A** — temperature + pressure isobars only (clean, fast)  
**Version B** — all weather layers: temp, pressure, precipitation, wind barbs  

Both use 12 h time steps for quick rendering. Run the full-quality cell at the bottom once you're happy with the layout.

In [7]:
# ── Version A: temperature + pressure only ───────────────────────────────────
animate_tracks_weather(
    df_apr26,
    era5_dir=ERA5_DIR,
    out_file="apr2026_A_temp_pressure.gif",
    color_col="species",

    show_pressure=True,
    show_temperature=True,
    show_precipitation=False,
    show_wind_barbs=False,

    snap_temp_to_midnight=True,
    isobar_interval=4,
    isobar_bold_interval=20,
    scale_bar_km=500,
    comet_trail_hours=360,

    time_resolution_hours=12,
    fps=6,
    width_px=900,
    height_px=620,
    dpi=100,
)

Loading tracks …
  4761 fixes | 93 individual(s) | 2026-04-01 – 2026-04-30
Loading ERA5 single-level data …
  Loading 3 ERA5 file(s) (cfgrib) …
  Dropped 32 NaN-only boundary time steps
  ERA5 variables: ['sp', 'msl', 'tcc', 'u10', 'v10', 't2m', 'u100', 'v100', 'tp', 'cbh', 'i10fg']
  ERA5 time:      2026-03-01 00:00 – 2026-05-29 15:00  (2152 steps)
  Computing temperature range …
  t2m range: 1.1 – 24.4 °C
  MSLP range: 998 – 1031 hPa, 10 isobar levels
  60 frames at 12h resolution
Rendering 60 frames …


Animating:   0%|          | 0/60 [00:00<?, ?frame/s]

  [dbg] t2m shape=(90, 134) min=-13.1 max=19.0 °C  levels=[ 1.06216431 24.41372681]
  [dbg] msl=shape=(90, 134) range=994–1035 hPa
  Frame    1/60: 2026-04-01 01:00 UTC


Animating:  33%|███▎      | 20/60 [00:09<00:22,  1.78frame/s]

  Frame   21/60: 2026-04-11 01:00 UTC


...

Animating:  98%|█████████▊| 59/60 [00:45<00:01,  1.16s/frame]

  Frame   60/60: 2026-04-30 13:00 UTC


Animating: 100%|██████████| 60/60 [00:46<00:00,  1.28frame/s]


Saved: apr2026_A_temp_pressure.gif


'apr2026_A_temp_pressure.gif'

In [8]:
# ── Version B: all weather layers ────────────────────────────────────────────
animate_tracks_weather(
    df_apr26,
    era5_dir=ERA5_DIR,
    out_file="apr2026_B_all_weather.gif",
    color_col="species",

    show_pressure=True,
    show_temperature=True,
    show_precipitation=True,
    show_wind_barbs=True,

    snap_temp_to_midnight=True,
    isobar_interval=4,
    isobar_bold_interval=20,
    wind_density=8,
    scale_bar_km=500,
    comet_trail_hours=360,

    time_resolution_hours=12,
    fps=6,
    width_px=900,
    height_px=620,
    dpi=100,
)

Loading tracks …
  4761 fixes | 93 individual(s) | 2026-04-01 – 2026-04-30
Loading ERA5 single-level data …
  Loading 3 ERA5 file(s) (cfgrib) …
  Dropped 32 NaN-only boundary time steps
  ERA5 variables: ['sp', 'msl', 'tcc', 'u10', 'v10', 't2m', 'u100', 'v100', 'tp', 'cbh', 'i10fg']
  ERA5 time:      2026-03-01 00:00 – 2026-05-29 15:00  (2152 steps)
  Computing temperature range …
  t2m range: 1.1 – 24.4 °C
  MSLP range: 998 – 1031 hPa, 10 isobar levels
  60 frames at 12h resolution
Rendering 60 frames …


Animating:   0%|          | 0/60 [00:00<?, ?frame/s]

  [dbg] t2m shape=(90, 134) min=-13.1 max=19.0 °C  levels=[ 1.06216431 24.41372681]
  [dbg] msl=shape=(90, 134) range=994–1035 hPa
  Frame    1/60: 2026-04-01 01:00 UTC


Animating:  33%|███▎      | 20/60 [00:09<00:24,  1.61frame/s]

  Frame   21/60: 2026-04-11 01:00 UTC


...

Animating:  98%|█████████▊| 59/60 [00:45<00:01,  1.16s/frame]

  Frame   60/60: 2026-04-30 13:00 UTC


Animating: 100%|██████████| 60/60 [00:47<00:00,  1.27frame/s]


Saved: apr2026_B_all_weather.gif


'apr2026_B_all_weather.gif'

In [14]:
# ── April 2026 — full-quality MP4 (uncomment when draft looks good) ──────────
# animate_tracks_weather(
#     df_apr26,
#     era5_dir=ERA5_DIR,
#     out_file="apr2026_full.mp4",
#     color_col="species",
#
#     show_pressure=True,
#     show_temperature=True,
#     show_precipitation=True,
#     show_wind_barbs=True,
#
#     snap_temp_to_midnight=True,
#     isobar_interval=4,
#     isobar_bold_interval=20,
#     wind_density=8,
#     scale_bar_km=500,
#     comet_trail_hours=360,
#
#     time_resolution_hours=6,
#     fps=10,
#     width_px=1400,
#     height_px=900,
#     dpi=120,
#     video_bitrate=8000,
# )

## Spring 2026 — full migration (Mar–May)

ERA5 files needed: `era5_single_2026_03`, `_04`, `_05` — all present on the network share.

In [15]:
# ── Spring 2026 draft (12h steps) ────────────────────────────────────────────
animate_tracks_weather(
    df_spr26,
    era5_dir=ERA5_DIR,
    out_file="spring2026_draft_2fps.pdf",
    color_col="species",

    show_pressure=True,
    show_temperature=True,
    show_precipitation=True,
    show_wind_barbs=False,

    snap_temp_to_midnight=True,
    isobar_interval=4,
    isobar_bold_interval=20,
    scale_bar_km=500,
    comet_trail_hours=480,       # longer tail — spring spans 3 months

    time_resolution_hours=12,
    fps=2,
    width_px=1000,
    height_px=680,
    dpi=100,
)

Loading tracks …
  6259 fixes | 94 individual(s) | 2026-03-19 – 2026-05-06
Loading ERA5 single-level data …
  Loading 3 ERA5 file(s) (cfgrib) …
  Dropped 32 NaN-only boundary time steps
  ERA5 variables: ['sp', 'msl', 'tcc', 'u10', 'v10', 't2m', 'u100', 'v100', 'tp', 'cbh', 'i10fg']
  ERA5 time:      2026-03-01 00:00 – 2026-05-29 15:00  (2152 steps)
  Computing temperature range …
  t2m range: 0.9 – 24.3 °C
  MSLP range: 998 – 1031 hPa, 10 isobar levels
  97 frames at 12h resolution
Rendering 97 frames …


Animating:   0%|          | 0/97 [00:00<?, ?frame/s]

  [dbg] t2m shape=(91, 141) min=-9.3 max=17.2 °C  levels=[ 0.85073853 24.31900024]
  [dbg] msl=shape=(91, 141) range=1003–1031 hPa
  Frame    1/97: 2026-03-19 12:00 UTC


Animating:  21%|██        | 20/97 [00:08<00:37,  2.03frame/s]

  Frame   21/97: 2026-03-29 12:00 UTC


...

Animating:  99%|█████████▉| 96/97 [01:16<00:01,  1.31s/frame]

  Frame   97/97: 2026-05-06 12:00 UTC


Animating: 100%|██████████| 97/97 [01:18<00:00,  1.24frame/s]


Saved: spring2026_draft_2fps.pdf


'spring2026_draft_2fps.pdf'

In [16]:
# ── Spring 2026 full-quality MP4 (uncomment when draft looks good) ────────────
# animate_tracks_weather(
#     df_spr26,
#     era5_dir=ERA5_DIR,
#     out_file="spring2026_full.mp4",
#     color_col="species",
#
#     show_pressure=True,
#     show_temperature=True,
#     show_precipitation=True,
#     show_wind_barbs=True,
#
#     snap_temp_to_midnight=True,
#     isobar_interval=4,
#     isobar_bold_interval=20,
#     wind_density=8,
#     scale_bar_km=500,
#     comet_trail_hours=480,
#
#     time_resolution_hours=6,
#     fps=12,
#     width_px=1400,
#     height_px=900,
#     dpi=120,
#     video_bitrate=8000,
# )

## Spring 2025 — full migration (Mar–May)

ERA5 files needed: `era5_single_2025_03`, `_04`, `_05` — all present on the network share.

In [17]:
# ── Spring 2025 draft (12h steps) ────────────────────────────────────────────
animate_tracks_weather(
    df_spr25,
    era5_dir=ERA5_DIR,
    out_file="spring2025_draft_3fps.gif",
    color_col="species",

    show_pressure=True,
    show_temperature=True,
    show_precipitation=True,
    show_wind_barbs=False,

    snap_temp_to_midnight=True,
    isobar_interval=4,
    isobar_bold_interval=20,
    scale_bar_km=500,
    comet_trail_hours=480,

    time_resolution_hours=12,
    fps=3,
    width_px=1000,
    height_px=680,
    dpi=100,
)

Loading tracks …
  19906 fixes | 201 individual(s) | 2025-03-18 – 2025-05-31
Loading ERA5 single-level data …
  Loading 4 ERA5 file(s) (cfgrib) …
  Dropped 48 NaN-only boundary time steps
  ERA5 variables: ['sp', 'msl', 'tcc', 'u10', 'v10', 't2m', 'u100', 'v100', 'tp', 'cbh', 'i10fg']
  ERA5 time:      2025-03-01 00:00 – 2025-06-30 23:00  (2928 steps)
  Computing temperature range …
  t2m range: 0.8 – 28.2 °C
  MSLP range: 998 – 1031 hPa, 10 isobar levels
  149 frames at 12h resolution
Rendering 149 frames …


Animating:   0%|          | 0/149 [00:00<?, ?frame/s]

  [dbg] t2m shape=(86, 135) min=-12.3 max=16.9 °C  levels=[ 0.84170532 28.18399048]
  [dbg] msl=shape=(86, 135) range=1008–1033 hPa
  Frame    1/149: 2025-03-18 13:00 UTC


Animating:  13%|█▎        | 20/149 [00:11<01:32,  1.40frame/s]

  Frame   21/149: 2025-03-28 13:00 UTC


...

Animating:  99%|█████████▉| 148/149 [04:06<00:02,  2.30s/frame]

  Frame  149/149: 2025-05-31 13:00 UTC


Animating: 100%|██████████| 149/149 [04:09<00:00,  1.67s/frame]


Saved: spring2025_draft_3fps.gif


'spring2025_draft_3fps.gif'

In [18]:
# ── Spring 2025 full-quality MP4 (uncomment when draft looks good) ────────────
# animate_tracks_weather(
#     df_spr25,
#     era5_dir=ERA5_DIR,
#     out_file="spring2025_full.mp4",
#     color_col="species",
#
#     show_pressure=True,
#     show_temperature=True,
#     show_precipitation=True,
#     show_wind_barbs=True,
#
#     snap_temp_to_midnight=True,
#     isobar_interval=4,
#     isobar_bold_interval=20,
#     wind_density=8,
#     scale_bar_km=500,
#     comet_trail_hours=480,
#
#     time_resolution_hours=6,
#     fps=12,
#     width_px=1400,
#     height_px=900,
#     dpi=120,
#     video_bitrate=8000,
# )